# Réseaux de neurones récurrents

Dans le module précédent, nous avons abordé les représentations sémantiques riches du texte. L’architecture que nous avons utilisée capture le sens global des mots dans une phrase, mais ne prend pas en compte **l’ordre** des mots, car l’opération d’agrégation qui suit les embeddings supprime cette information du texte original. Parce que ces modèles ne peuvent pas représenter l’ordre des mots, ils ne peuvent pas résoudre des tâches plus complexes ou ambiguës telles que la génération de texte ou la réponse à des questions.

Pour capturer le sens d’une séquence de texte, nous allons utiliser une architecture de réseau de neurones appelée **réseau de neurones récurrent**, ou RNN. Lors de l’utilisation d’un RNN, nous faisons passer notre phrase dans le réseau un jeton à la fois, et le réseau produit un certain **état**, que nous transmettons ensuite au réseau avec le jeton suivant.

![Image montrant un exemple de génération par un réseau de neurones récurrent.](../../../../../translated_images/fr/rnn.27f5c29c53d727b5.webp)

Étant donné la séquence d’entrée de jetons $X_0,\dots,X_n$, le RNN crée une séquence de blocs de réseau de neurones, et entraîne cette séquence de bout en bout en utilisant la rétropropagation. Chaque bloc de réseau prend une paire $(X_i,S_i)$ en entrée, et produit $S_{i+1}$ comme résultat. L’état final $S_n$ ou la sortie $Y_n$ est envoyé dans un classificateur linéaire pour produire le résultat. Tous les blocs de réseau partagent les mêmes poids, et sont entraînés de bout en bout en utilisant une seule passe de rétropropagation.

> La figure ci-dessus montre un réseau de neurones récurrent sous forme déroulée (à gauche), et dans une représentation récurrente plus compacte (à droite). Il est important de réaliser que toutes les cellules RNN ont les mêmes **poids partageables**.

Parce que les vecteurs d’état $S_0,\dots,S_n$ sont transmis à travers le réseau, le RNN est capable d’apprendre des dépendances séquentielles entre les mots. Par exemple, lorsque le mot *not* apparaît quelque part dans la séquence, il peut apprendre à nier certains éléments dans le vecteur d’état.

À l’intérieur, chaque cellule RNN contient deux matrices de poids : $W_H$ et $W_I$, et un biais $b$. À chaque étape du RNN, donné l’entrée $X_i$ et l’état d’entrée $S_i$, l’état de sortie est calculé comme $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, où $f$ est une fonction d’activation (souvent $\tanh$).

> Pour des problèmes comme la génération de texte (que nous aborderons dans l’unité suivante) ou la traduction automatique, nous voulons aussi obtenir une valeur de sortie à chaque étape du RNN. Dans ce cas, il y a aussi une autre matrice $W_O$, et la sortie est calculée comme $Y_i=f(W_O\times S_i+b_O)$.

Voyons comment les réseaux de neurones récurrents peuvent nous aider à classer notre ensemble de données de nouvelles.

> Pour l’environnement bac à sable, nous devons exécuter la cellule suivante pour nous assurer que la bibliothèque requise est installée, et que les données sont préchargées. Si vous exécutez localement, vous pouvez ignorer la cellule suivante.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Lors de l'entraînement de grands modèles, l'allocation de mémoire GPU peut devenir un problème. Nous pouvons également avoir besoin d'expérimenter avec différentes tailles de minibatch, de sorte que les données tiennent dans notre mémoire GPU, tout en assurant une rapidité suffisante de l'entraînement. Si vous exécutez ce code sur votre propre machine GPU, vous pouvez essayer d'ajuster la taille du minibatch pour accélérer l'entraînement.

> **Note** : Certaines versions des pilotes NVidia sont connues pour ne pas libérer la mémoire après l'entraînement du modèle. Nous exécutons plusieurs exemples dans ce carnet, ce qui pourrait entraîner un épuisement de la mémoire dans certaines configurations, notamment si vous effectuez vos propres expériences dans le même carnet. Si vous rencontrez des erreurs étranges au début de l'entraînement du modèle, il peut être utile de redémarrer le noyau du carnet.


In [3]:
batch_size = 16
embed_size = 64

## Classificateur RNN simple

Dans le cas d'un RNN simple, chaque unité récurrente est un réseau linéaire simple, qui prend en entrée un vecteur d'entrée et un vecteur d'état, et produit un nouveau vecteur d'état. Dans Keras, cela peut être représenté par la couche `SimpleRNN`.

Bien que nous puissions passer des jetons encodés en one-hot directement à la couche RNN, ce n'est pas une bonne idée en raison de leur grande dimensionnalité. Par conséquent, nous utiliserons une couche d'embedding pour réduire la dimensionnalité des vecteurs de mots, suivie d'une couche RNN, et enfin d'un classificateur `Dense`.

> **Note** : Dans les cas où la dimensionnalité n'est pas si élevée, par exemple lors de la tokenisation au niveau des caractères, il peut être judicieux de passer directement des jetons encodés en one-hot dans la cellule RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Note :** Nous utilisons ici une couche d'embedding non entraînée pour simplifier, mais pour de meilleurs résultats, nous pouvons utiliser une couche d'embedding pré-entraînée avec Word2Vec, comme décrit dans l'unité précédente. Ce serait un bon exercice pour vous d'adapter ce code afin de travailler avec des embeddings pré-entraînés.

Maintenant, entraînons notre RNN. Les RNN en général sont assez difficiles à entraîner, car une fois que les cellules RNN sont déroulées sur la longueur de la séquence, le nombre de couches impliquées dans la rétropropagation est assez important. Ainsi, nous devons sélectionner un taux d'apprentissage plus faible et entraîner le réseau sur un jeu de données plus large pour obtenir de bons résultats. Cela peut prendre un certain temps, donc l'utilisation d'un GPU est préférable.

Pour accélérer les choses, nous allons entraîner le modèle RNN uniquement sur les titres de news, en omettant la description. Vous pouvez essayer d'entraîner avec la description et voir si vous parvenez à faire entraîner le modèle.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Note** que la précision est probablement plus faible ici, car nous entraînons uniquement sur des titres de presse.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Maintenant que nous utilisons le masquage, nous pouvons entraîner le modèle sur l'ensemble du jeu de données des titres et descriptions.

> **Note** : Avez-vous remarqué que nous avons utilisé un vectoriseur entraîné sur les titres de nouvelles, et non sur le corps entier de l'article ? Cela peut potentiellement entraîner l'ignorance de certains tokens, il est donc préférable de réentraîner le vectoriseur. Cependant, cela pourrait n'avoir qu'un effet très faible, donc nous allons conserver le vectoriseur pré-entraîné précédent pour des raisons de simplicité.


## LSTM : mémoire à long terme

L'un des principaux problèmes des RNN est le **problème des gradients qui disparaissent**. Les RNN peuvent être assez longs et avoir du mal à propager les gradients jusqu'à la première couche du réseau lors de la rétropropagation. Lorsque cela se produit, le réseau ne peut pas apprendre les relations entre des tokens distants. Une façon d'éviter ce problème est d'introduire une **gestion explicite de l'état** en utilisant des **portes**. Les deux architectures les plus courantes qui introduisent des portes sont la **mémoire à long terme** (LSTM) et l'**unité à relais à portes** (GRU). Nous aborderons ici les LSTMs.

![Image montrant un exemple de cellule de mémoire à long terme](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Un réseau LSTM est organisé de manière similaire à un RNN, mais il y a deux états qui sont transmis de couche en couche : l'état actuel $c$, et le vecteur caché $h$. À chaque unité, le vecteur caché $h_{t-1}$ est combiné avec l'entrée $x_t$, et ensemble ils contrôlent ce qui arrive à l'état $c_t$ et à la sortie $h_{t}$ via les **portes**. Chaque porte a une activation sigmoidale (sortie dans la plage $[0,1]$), que l’on peut considérer comme un masque bit à bit lorsqu'il est multiplié par le vecteur d'état. Les LSTMs possèdent les portes suivantes (de gauche à droite sur l'image ci-dessus) :
* **porte d’oubli** qui détermine quelles composantes du vecteur $c_{t-1}$ nous devons oublier, et lesquelles transmettre.
* **porte d’entrée** qui détermine combien d’informations provenant du vecteur d’entrée et du vecteur caché précédent doivent être incorporées dans le vecteur d’état.
* **porte de sortie** qui prend le nouveau vecteur d’état et décide quelles de ses composantes seront utilisées pour produire le nouveau vecteur caché $h_t$.

Les composantes de l'état $c$ peuvent être vues comme des indicateurs qu'on peut activer ou désactiver. Par exemple, lorsque nous rencontrons le nom *Alice* dans la séquence, nous supposons qu'il s'agit d'une femme, et activons le drapeau dans l'état qui indique que nous avons un nom féminin dans la phrase. Lorsque nous rencontrons ensuite les mots *and Tom*, nous activerons le drapeau indiquant que nous avons un nom au pluriel. Ainsi, en manipulant l’état, nous pouvons suivre les propriétés grammaticales de la phrase.

> **Note** : Voici une excellente ressource pour comprendre le fonctionnement interne des LSTMs : [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) par Christopher Olah.

Bien que la structure interne d'une cellule LSTM puisse sembler complexe, Keras cache cette implémentation à l'intérieur de la couche `LSTM`, donc la seule chose que nous devons faire dans l'exemple ci-dessus est de remplacer la couche récurrente :


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Note** que l'entraînement des LSTM est également assez lent, et vous ne remarquerez peut-être pas une grande augmentation de la précision au début de l'entraînement. Vous devrez peut-être continuer l'entraînement pendant un certain temps pour atteindre une bonne précision.


## RNN bidirectionnels et multi-couches

Dans nos exemples jusqu'à présent, les réseaux récurrents fonctionnent du début d'une séquence jusqu'à la fin. Cela nous semble naturel car cela suit la même direction dans laquelle nous lisons ou écoutons un discours. Cependant, pour les scénarios qui nécessitent un accès aléatoire à la séquence d'entrée, il est plus logique d'exécuter le calcul récurrent dans les deux sens. Les RNN qui permettent des calculs dans les deux directions s'appellent des RNN **bidirectionnels**, et ils peuvent être créés en enveloppant la couche récurrente avec une couche spéciale `Bidirectonal`.

> **Note** : La couche `Bidirectional` crée deux copies de la couche qu'elle contient, et définit la propriété `go_backwards` de l'une de ces copies à `True`, ce qui la fait fonctionner dans la direction opposée le long de la séquence.

Les réseaux récurrents, unidirectionnels ou bidirectionnels, capturent des motifs au sein d'une séquence, et les stockent sous forme de vecteurs d'état ou les renvoient en sortie. Comme avec les réseaux convolutionnels, nous pouvons construire une autre couche récurrente suivant la première pour capturer des motifs de niveau supérieur, construits à partir des motifs de niveau inférieur extraits par la première couche. Cela nous conduit à la notion de **RNN multi-couches**, qui consiste en deux réseaux récurrents ou plus, où la sortie de la couche précédente est transmise à la couche suivante en entrée.

![Image montrant un RNN à mémoire à long terme multilayer](../../../../../translated_images/fr/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Image tirée de [ce merveilleux article](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) de Fernando López.*

Keras facilite la construction de ces réseaux, car vous avez juste besoin d'ajouter plus de couches récurrentes au modèle. Pour toutes les couches sauf la dernière, nous devons spécifier le paramètre `return_sequences=True`, car nous avons besoin que la couche renvoie tous les états intermédiaires, et pas seulement l'état final du calcul récurrent.

Construisons un LSTM bidirectionnel à deux couches pour notre problème de classification.

> **Note** ce code prend encore assez longtemps à s'exécuter, mais il nous donne la meilleure précision que nous ayons vue jusqu'ici. Donc peut-être que cela vaut la peine d'attendre et de voir le résultat.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs pour d'autres tâches

Jusqu'à présent, nous nous sommes concentrés sur l'utilisation des RNN pour classifier des séquences de texte. Mais ils peuvent gérer bien d'autres tâches, telles que la génération de texte et la traduction automatique &mdash; nous étudierons ces tâches dans l'unité suivante.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
